#### 1. Import the libraries

In [30]:
import pandas as pd

## Exercise 1: Build the CSVReader Class (OOP Foundation)

In [31]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.data = None
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df
    def preview(self, n=5):
        print(self.df.head(n))

# Test 
reader = CSVReader("sample_data.csv")
reader.read()
reader.preview(5)

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Exercise 2: Strategy Pattern for Missing Values

In [32]:
from abc import ABC, abstractmethod
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df):
        pass

class DropMissing(MissingValueStrategy):
    def handle(self, df):
        return df.dropna()

class FillMean(MissingValueStrategy):
    def handle(self, df):
        return df.fillna(df.mean(numeric_only=True))

class FillMode(MissingValueStrategy):
    def handle(self, df):
        return df.fillna(df.mode().iloc[0])

class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def clean(self, df):
        return self.strategy.handle(df)

# Test 
df = pd.read_csv("sample_data.csv")

cleaner = DataCleaner(FillMode())
print(cleaner.clean(df))

   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0


## Exercise 3: Add Decorators for Logging and Timing

### 3.2. Create the @log_action decorator

- time: Measure execution time.
- wraps: Preserve the original function's information.

In [33]:
import time
from functools import wraps

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting: {func.__name__}")

        result = func(*args, **kwargs)

        print(f"Finished: {func.__name__}")
        return result

    return wrapper

- func: The original method that we decorate.

- wrapper: A new function that adds extra behavior.

- *args, **kwargs: Accept the arguments of the original method.

- result = func(*args, **kwargs): Execute the original method.

- return result: Return its result.

### 3.3. Create the @log_time decorator

In [34]:
def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()

        result = func(*args, **kwargs)

        end = time.perf_counter()

        print(f"Execution time: {end - start:.4f} seconds")
        return result

    return wrapper

- start: Record the time before execution.

- result: Run the original method.

- end: Record the time after execution.

- end - start: Calculate how long the method took.

### 3.4. Apply decorators to CSVReader

In [35]:
class CSVReader:
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None

    @log_time
    @log_action
    def read(self):
        self.df = pd.read_csv(self.file_path)
        return self.df

    def preview(self, n=5):
        print(self.df.head(n))

### 3.5. Test using sample_data.csv

In [36]:
reader = CSVReader("sample_data.csv")

reader.read()
reader.preview(5)

Starting: read
Finished: read
Execution time: 0.0025 seconds
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


## Exercise 4: Implement Factory Pattern for Data Transformations

In [37]:
class DataTransform(ABC):
    @abstractmethod
    def apply(self, df):
        pass

class NormalizeColumns(DataTransform):
    def apply(self, df):
        df = df.copy()
        df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
        return df

class RemoveDuplicates(DataTransform):
    def apply(self, df):
        return df.drop_duplicates()

class  StandardizeText(DataTransform):
    def apply(self, df):
        df = df.copy()
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip().str.lower()
        return df

Create TransformFactory

In [38]:
class TransformFactory:
    @staticmethod
    def create(transform_type):
        if transform_type == "normalize":
            return NormalizeColumns()
        elif transform_type == "duplicates":
            return RemoveDuplicates()
        elif transform_type == "text":
            return StandardizeText()
        else:
            raise ValueError(f"Unknown transform type: {transform_type}")

Testing

In [39]:
data = {
    "Full Name": [" Dara ", "SOKHA", " Dara "],
    "Age": [20, 21, 20]
}

df = pd.DataFrame(data)

transform = TransformFactory.create("normalize")
df = transform.apply(df)

transform = TransformFactory.create("text")
df = transform.apply(df)

transform = TransformFactory.create("duplicates")
df = transform.apply(df)

print(df)

  full_name  age
0      dara   20
1     sokha   21


## Exercise 5: Build a Full Cleaning Pipeline 

In [40]:
class DataPipeline(ABC):
    def run(self):
        df = self.load()
        df = self.clean(df)
        df = self.transform(df)
        self.save(df)
        return df

    @abstractmethod
    def load(self):
        pass

class CSVDataPipeline(DataPipeline):
    def __init__(self, input_file, output_file, strategy):
        self.input_file = input_file
        self.output_file = output_file
        self.strategy = strategy

    def load(self):
        reader = CSVReader(self.input_file)
        return reader.read()

    def clean(self, df):
        cleaner = DataCleaner(self.strategy)
        return cleaner.clean(df)

    def transform(self, df):
        for name in ["normalize", "text", "duplicates"]:
            transform = TransformFactory.create(name)
            df = transform.apply(df)
        return df

    def save(self, df):
        df.to_csv(self.output_file, index=False)
        print(f"Data saved to {self.output_file}")

In [41]:
# Test 

pipeline = CSVDataPipeline(
    input_file="sample_data.csv",
    output_file="cleaned_data.csv",
    strategy=FillMode()
)

result = pipeline.run()

print(result)

Starting: read
Finished: read
Execution time: 0.0012 seconds
Data saved to cleaned_data.csv
   id     name   age  height_cm  weight_kg         city  score
0   1    alice  29.0      165.0       68.0     new york   85.0
1   2      bob  27.0      172.0       55.0  los angeles   90.0
2   3  charlie  35.0      168.0       72.0      chicago   75.0
3   4    david  27.0      160.0       80.0      houston   75.0
4   5      eva  27.0      160.0       55.0     new york   88.0
5   6    frank  30.0      175.0       85.0  los angeles   75.0
6   7    grace  27.0      162.0       60.0      chicago   92.0
